In [ ]:
"""
----
가장 높은 Kaggle 점수(0.99788)를 기록한 모델(exp_stratified_v4_customaug)의 학습 스크립트.
"""

In [ ]:
import numpy as np
import torch
import albumentations as A
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.data.dataset import YOLODataset

In [ ]:
def build_pixel_transforms() -> A.Compose:
    return A.Compose(
        [
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
            A.RandomBrightnessContrast(
                brightness_limit=0.15, contrast_limit=0.2, p=0.3
            ),
            A.ImageCompression(quality_lower=70, quality_upper=95, p=0.2),
        ]
    )


class PillYOLODataset(YOLODataset):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pixel_transform = build_pixel_transforms()

    def __getitem__(self, index):
        sample = super().__getitem__(index)

        img = sample["img"].permute(1, 2, 0).numpy()
        img = np.ascontiguousarray(img)

        transformed = self.pixel_transform(image=img)
        img = transformed["image"]

        sample["img"] = torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1)
        return sample


class PillTrainer(DetectionTrainer):
    def build_dataset(self, img_path, mode="train", batch=None):
        dataset = super().build_dataset(img_path, mode, batch)
        if mode == "train":
            dataset.__class__ = PillYOLODataset
            dataset.pixel_transform = build_pixel_transforms()
        return dataset


DATA_YAML = "./data/dataset_stratified_v4/data.yaml"
PROJECT_DIR = "./runs"
RUN_NAME = "exp_stratified_v4_customaug"

In [ ]:
def main():
    model = YOLO("yolo11n.pt")
    model.train(
        trainer=PillTrainer,
        data=DATA_YAML,
        epochs=60,
        patience=20,
        batch=16,
        imgsz=640,
        optimizer="AdamW",
        device=0,
        workers=2,
        lr0=0.00176,
        weight_decay=0.0,
        momentum=0.9349,
        box=7.70337,
        cls=0.57286,
        dfl=2.02811,
        mosaic=0.85211,
        mixup=0.00489,
        flipud=0.00328,
        fliplr=0.0,
        hsv_h=0.0171,
        hsv_s=0.60513,
        hsv_v=0.44902,
        translate=0.0,
        scale=0.00162,
        erasing=0.0,
        auto_augment="none",
        augment=True,
        project=PROJECT_DIR,
        name=RUN_NAME,
        exist_ok=True,
    )

In [ ]:
main()

In [ ]:
best_model = YOLO(f"{PROJECT_DIR}/{RUN_NAME}/weights/best.pt")
results = best_model.val(data=DATA_YAML, split="val")

print(f"mAP@50: {results.box.map50:.4f}")
print(f"mAP@50-95: {results.box.map:.4f}")
print(f"Precision: {results.box.mp:.4f}")
print(f"Recall: {results.box.mr:.4f}")